In [1]:
import numpy as np
import pandas as pd

In [2]:
indice_analfabetismo = pd.read_csv('../data/outras_fontes/ipeadata[27-08-2026-11-29]-indice-analfabetismo-mun.csv', sep=';', skiprows=1)
indice_analfabetismo.head()

,Sigla,Código,Município,2022,Unnamed: 4
0,AC,1200013,Acrelândia,"11,65",NaN
1,AC,1200054,Assis Brasil,"14,7",NaN
2,AC,1200104,Brasiléia,"10,99",NaN
3,AC,1200138,Bujari,"18,74",NaN
4,AC,1200179,Capixaba,"15,73",NaN


In [3]:
#Conferir linhas em branco
indice_analfabetismo.isna().sum()

Sigla            0
Código           0
Município        0
2022            27
Unnamed: 4    5597
dtype: int64

In [4]:
indice_analfabetismo['Unnamed: 4'].unique()

array([nan])

In [5]:
# Excluir coluna 'Unnamed: 4'
indice_analfabetismo.drop(columns='Unnamed: 4', inplace=True)
indice_analfabetismo.head()

,Sigla,Código,Município,2022
0,AC,1200013,Acrelândia,"11,65"
1,AC,1200054,Assis Brasil,"14,7"
2,AC,1200104,Brasiléia,"10,99"
3,AC,1200138,Bujari,"18,74"
4,AC,1200179,Capixaba,"15,73"


In [6]:
# Alterar o nome da coluna '2022' para 'indice_analf'
indice_analfabetismo['indice_analf'] = indice_analfabetismo['2022']
indice_analfabetismo.drop(columns = '2022', inplace = True)
indice_analfabetismo.head()

,Sigla,Código,Município,indice_analf
0,AC,1200013,Acrelândia,"11,65"
1,AC,1200054,Assis Brasil,"14,7"
2,AC,1200104,Brasiléia,"10,99"
3,AC,1200138,Bujari,"18,74"
4,AC,1200179,Capixaba,"15,73"


In [7]:
# Entender as taxas em branco
indice_analfabetismo[indice_analfabetismo['indice_analf'].isna()][['Município','Código','Sigla']]

,Município,Código,Sigla
619,Trancoso,2999919,BA
620,Barcellos,2999929,BA
621,Olivença,2999939,BA
622,Villa Verde,2999949,BA
757,Cococi,2310298,CE
808,Entre Rios,2399919,CE
809,Mecejana,2399929,CE
810,Vertentes,2399939,CE
890,Riacho,3299919,ES
891,Ponte do Itabapoana,3299929,ES


27 municípios não possuem a taxa de analfabetismo nesta pesquisa de 2022. Como não existe uma relação entre estes municípios, pois não repete a UF, podemos pensar numa forma de completar estes dados.

Escolhi usar a mediana das notas da UF, pois representa melhor o ponto central do índice do Estado.

In [8]:
# Alterando a coluna 'indice_analf' para o tipo float
indice_analfabetismo['indice_analf'] = indice_analfabetismo['indice_analf'].str.replace(',', '.', regex=False)
indice_analfabetismo['indice_analf'] = indice_analfabetismo['indice_analf'].str.strip() 
indice_analfabetismo['indice_analf'] = indice_analfabetismo['indice_analf'].astype(float)

In [9]:
# 1. Calcula a mediana do índice de analfabetismo agrupada por 'Estado'
mediana_por_grupo = round(indice_analfabetismo.groupby('Sigla')['indice_analf'].median(),2)

# 2. Preenche os valores nulos com o resultado correspondente ao grupo de cada linha
indice_analfabetismo['indice_analf'] = indice_analfabetismo['indice_analf'].fillna(indice_analfabetismo['Sigla'].map(mediana_por_grupo))

print("\n--- DataFrame com Nulos Preenchidos ---")
print(indice_analfabetismo)
print(f'Quantidade de dados nulos: {indice_analfabetismo.isna().sum().sum()}')


--- DataFrame com Nulos Preenchidos ---
     Sigla   Código       Município  indice_analf
0       AC  1200013      Acrelândia         11.65
1       AC  1200054    Assis Brasil         14.70
2       AC  1200104       Brasiléia         10.99
3       AC  1200138          Bujari         18.74
4       AC  1200179        Capixaba         15.73
...    ...      ...             ...           ...
5592    TO  1721208  Tocantinópolis          9.96
5593    TO  1721257        Tupirama         12.03
5594    TO  1721307      Tupiratins         15.11
5595    TO  1722081    Wanderlândia         13.61
5596    TO  1722107         Xambioá         13.62

[5597 rows x 4 columns]
Quantidade de dados nulos: 0


In [10]:
#Testar se funcionou a substituição

print(indice_analfabetismo[indice_analfabetismo['Código']== 2999919])
print('---------------------------------------------')
print(mediana_por_grupo[mediana_por_grupo.index=='BA'])

    Sigla   Código Município  indice_analf
619    BA  2999919  Trancoso         18.39
---------------------------------------------
Sigla
BA    18.39
Name: indice_analf, dtype: float64


In [11]:
# Salvar em formato parquet para agregar na base do modelo
indice_analfabetismo.to_parquet('../data/outras_fontes/indice_analfabetismo.parquet')

Próxima base - Rendimento domiciliar per capita médio (SIS/IBGE)

In [12]:
rendimento_dom_per_capta = pd.read_csv('../data/outras_fontes/ipeadata[27-08-2026-11-39]-rendimento-domiciliar-per-capta.csv', sep=';', skiprows=1)
rendimento_dom_per_capta.head()

,Sigla,Código,Estado,2020,2021,2022,2023,2024,Unnamed: 8
0,AC,12,Acre,"1122,50467904204","1011,46758708461","1073,66464167404","1074,35495100746","1258,98023431041",NaN
1,AL,27,Alagoas,"955,443555186744","869,463907257867","961,281623320264","1102,35916028865","1317,31195720455",NaN
2,AM,13,Amazonas,"1009,46120042836","914,875913795966","990,727737612939","1165,82025270929","1229,9496248055",NaN
3,AP,16,Amapá,"1042,88169792415","953,255514530622","1217,73481079677","1491,75918304983","1508,20788450962",NaN
4,BA,29,Bahia,"1181,66385983687","981,299041851291","1046,79608531868","1128,86495578586","1340,74374032185",NaN


In [13]:
# Vamos usar apenas a informação de 2023

rendimento_dom_per_capta = rendimento_dom_per_capta[['Sigla','Estado','2023']]
rendimento_dom_per_capta['rendimento_pc'] = rendimento_dom_per_capta['2023']

# Alterar o tipo para float
rendimento_dom_per_capta['rendimento_pc'] = rendimento_dom_per_capta['rendimento_pc'].str.replace(',', '.', regex=False)
rendimento_dom_per_capta['rendimento_pc'] = rendimento_dom_per_capta['rendimento_pc'].str.strip() 
rendimento_dom_per_capta['rendimento_pc'] = round(rendimento_dom_per_capta['rendimento_pc'].astype(float),2)

In [14]:
rendimento_dom_per_capta.describe()

,rendimento_pc
count,27.000000
mean,1638.999630
std,541.822611
min,968.530000
25%,1182.015000
50%,1491.760000
75%,1981.545000
max,3214.850000


In [15]:
# Salvar em formato parquet para agregar na base do modelo
rendimento_dom_per_capta.to_parquet('../data/outras_fontes/rendimento_dom_per_capta.parquet')

Próxima base - INSE (Indicador de Nível Socioeconômico)

In [41]:
inse = pd.read_csv('../data/outras_fontes/INSE_2023_municipios.csv', sep=';')
inse.head()

,NU_ANO_SAEB,CO_UF,SG_UF,NO_UF,CO_MUNICIPIO,NO_MUNICIPIO,TP_TIPO_REDE,TP_LOCALIZACAO,TP_CAPITAL,QTD_ALUNOS_INSE,MEDIA_INSE,PC_NIVEL_1,PC_NIVEL_2,PC_NIVEL_3,PC_NIVEL_4,PC_NIVEL_5,PC_NIVEL_6,PC_NIVEL_7,PC_NIVEL_8
0,2023,11,RO,Rondônia,1100122,Ji-Paraná,2,1,2,3258,"5,0083","0,16","6,71","18,43","25,63","24,64","14,36","9,18","0,89"
1,2023,11,RO,Rondônia,1101559,Teixeirópolis,0,2,2,19,"4,8242","5,26",NaN,"21,05","31,58","26,32","15,79",NaN,NaN
2,2023,11,RO,Rondônia,1100205,Porto Velho,3,1,1,3610,"4,8843","0,57","11,59",20,"26,07","20,78","12,87","7,11","1,02"
3,2023,11,RO,Rondônia,1101005,Governador Jorge Teixeira,5,2,2,46,"4,8115",0,"6,59","21,59","39,09","19,77","8,64","4,32",NaN
4,2023,11,RO,Rondônia,1100205,Porto Velho,2,1,1,9541,"5,0505","0,3","7,89","17,27","23,41","23,09","16,37","10,45","1,23"


In [42]:
inse.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125741 entries, 0 to 125740
Data columns (total 19 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   NU_ANO_SAEB      125741 non-null  int64 
 1   CO_UF            125741 non-null  int64 
 2   SG_UF            125741 non-null  object
 3   NO_UF            125741 non-null  object
 4   CO_MUNICIPIO     125741 non-null  int64 
 5   NO_MUNICIPIO     125741 non-null  object
 6   TP_TIPO_REDE     125741 non-null  int64 
 7   TP_LOCALIZACAO   125741 non-null  int64 
 8   TP_CAPITAL       125741 non-null  int64 
 9   QTD_ALUNOS_INSE  125741 non-null  int64 
 10  MEDIA_INSE       124851 non-null  object
 11  PC_NIVEL_1       125656 non-null  object
 12  PC_NIVEL_2       121281 non-null  object
 13  PC_NIVEL_3       124129 non-null  object
 14  PC_NIVEL_4       125221 non-null  object
 15  PC_NIVEL_5       124967 non-null  object
 16  PC_NIVEL_6       122791 non-null  object
 17  PC_NIVEL_7

In [43]:
# Alterar o tipo para float
inse['MEDIA_INSE'] = inse['MEDIA_INSE'].str.replace(',', '.', regex=False)
inse['MEDIA_INSE'] = inse['MEDIA_INSE'].str.strip() 
inse['MEDIA_INSE'] = inse['MEDIA_INSE'].astype(float)

In [44]:
# Verificando as colunas 
inse['NU_ANO_SAEB'].unique()

array([2023])

In [45]:
inse.isna().sum()

NU_ANO_SAEB            0
CO_UF                  0
SG_UF                  0
NO_UF                  0
CO_MUNICIPIO           0
NO_MUNICIPIO           0
TP_TIPO_REDE           0
TP_LOCALIZACAO         0
TP_CAPITAL             0
QTD_ALUNOS_INSE        0
MEDIA_INSE           890
PC_NIVEL_1            85
PC_NIVEL_2          4460
PC_NIVEL_3          1612
PC_NIVEL_4           520
PC_NIVEL_5           774
PC_NIVEL_6          2950
PC_NIVEL_7          7811
PC_NIVEL_8         42414
dtype: int64

In [46]:
# Vamos entender quais são as linhas com a média em branco
print(inse[inse['MEDIA_INSE'].isna()]['TP_TIPO_REDE'].unique())



[4]


In [47]:
#Todos os campos com a média nula tem o tipo de rede = 4
#Vamos ver a quantidade de linhas com o tipo de rede = 4
len(inse[inse['TP_TIPO_REDE'] == 4])

890

Todos as linhas com o tipo de rede 4 não possuem média.

In [48]:
inse['CO_MUNICIPIO'].nunique()

5558

Temos 125741 linhas para 5558 municípios. Vamos precisar agregar as informações para usar esta tabela com a base do modelo.

In [49]:
# Primeiro precisamos entender porque aparecem tantas linhas.
# Vou pegar um município qualquer para entender.
inse[inse['CO_MUNICIPIO']==1100122]

,NU_ANO_SAEB,CO_UF,SG_UF,NO_UF,CO_MUNICIPIO,NO_MUNICIPIO,TP_TIPO_REDE,TP_LOCALIZACAO,TP_CAPITAL,QTD_ALUNOS_INSE,MEDIA_INSE,PC_NIVEL_1,PC_NIVEL_2,PC_NIVEL_3,PC_NIVEL_4,PC_NIVEL_5,PC_NIVEL_6,PC_NIVEL_7,PC_NIVEL_8
0,2023,11,RO,Rondônia,1100122,Ji-Paraná,2,1,2,3258,5.0083,"0,16","6,71","18,43","25,63","24,64","14,36","9,18","0,89"
14,2023,11,RO,Rondônia,1100122,Ji-Paraná,2,1,2,3258,5.0083,"0,16","6,71","18,43","25,63","24,64","14,36","9,18","0,89"
18,2023,11,RO,Rondônia,1100122,Ji-Paraná,2,1,2,3258,5.0083,"0,16","6,71","18,43","25,63","24,64","14,36","9,18","0,89"
44,2023,11,RO,Rondônia,1100122,Ji-Paraná,2,1,2,3258,5.0083,"0,16","6,71","18,43","25,63","24,64","14,36","9,18","0,89"
80,2023,11,RO,Rondônia,1100122,Ji-Paraná,3,0,2,779,5.0084,"0,13","7,15","16,59","26,3","28,58","11,7","8,08","1,46"
81,2023,11,RO,Rondônia,1100122,Ji-Paraná,1,1,2,110,5.3650,0,"3,14","9,7","22,57","21,93","21,25","20,42","0,99"
83,2023,11,RO,Rondônia,1100122,Ji-Paraná,2,1,2,3258,5.0083,"0,16","6,71","18,43","25,63","24,64","14,36","9,18","0,89"
87,2023,11,RO,Rondônia,1100122,Ji-Paraná,2,1,2,3258,5.0083,"0,16","6,71","18,43","25,63","24,64","14,36","9,18","0,89"
88,2023,11,RO,Rondônia,1100122,Ji-Paraná,2,1,2,3258,5.0083,"0,16","6,71","18,43","25,63","24,64","14,36","9,18","0,89"
112,2023,11,RO,Rondônia,1100122,Ji-Paraná,2,1,2,3258,5.0083,"0,16","6,71","18,43","25,63","24,64","14,36","9,18","0,89"


In [50]:
inse = inse.drop_duplicates()
len(inse)

71576

In [51]:
inse[inse['CO_MUNICIPIO']==1100122]

,NU_ANO_SAEB,CO_UF,SG_UF,NO_UF,CO_MUNICIPIO,NO_MUNICIPIO,TP_TIPO_REDE,TP_LOCALIZACAO,TP_CAPITAL,QTD_ALUNOS_INSE,MEDIA_INSE,PC_NIVEL_1,PC_NIVEL_2,PC_NIVEL_3,PC_NIVEL_4,PC_NIVEL_5,PC_NIVEL_6,PC_NIVEL_7,PC_NIVEL_8
0,2023,11,RO,Rondônia,1100122,Ji-Paraná,2,1,2,3258,5.0083,"0,16","6,71","18,43","25,63","24,64","14,36","9,18","0,89"
80,2023,11,RO,Rondônia,1100122,Ji-Paraná,3,0,2,779,5.0084,"0,13","7,15","16,59","26,3","28,58","11,7","8,08","1,46"
81,2023,11,RO,Rondônia,1100122,Ji-Paraná,1,1,2,110,5.3650,0,"3,14","9,7","22,57","21,93","21,25","20,42","0,99"
208,2023,11,RO,Rondônia,1100122,Ji-Paraná,6,1,2,3997,5.0235,"0,16","6,74","17,89","25,5","24,84","14,18","9,66","1,03"
228,2023,11,RO,Rondônia,1100122,Ji-Paraná,3,1,2,629,5.0278,"0,16","7,72","16,8","25,52","26,66","11,51","9,82","1,81"
252,2023,11,RO,Rondônia,1100122,Ji-Paraná,3,2,2,150,4.9268,0,"4,76","15,72","29,57","36,64","12,5","0,81",NaN
342,2023,11,RO,Rondônia,1100122,Ji-Paraná,0,1,2,3997,5.0235,"0,16","6,74","17,89","25,5","24,84","14,18","9,66","1,03"
364,2023,11,RO,Rondônia,1100122,Ji-Paraná,6,0,2,4147,5.0203,"0,15","6,67","17,82","25,64","25,24","14,12","9,37","0,99"
395,2023,11,RO,Rondônia,1100122,Ji-Paraná,1,0,2,110,5.3650,0,"3,14","9,7","22,57","21,93","21,25","20,42","0,99"
603,2023,11,RO,Rondônia,1100122,Ji-Paraná,5,2,2,150,4.9268,0,"4,76","15,72","29,57","36,64","12,5","0,81",NaN


In [60]:
# Agora vamos agregar por município e tipo de rede, mantendo apenas a média INSE.
inse = inse.groupby(['NU_ANO_SAEB','CO_MUNICIPIO','TP_TIPO_REDE'], as_index=False)['MEDIA_INSE'].mean().round(3)

In [61]:
len(inse)

28978

In [62]:
# Salvar em formato parquet para agregar na base do modelo
inse.to_parquet('../data/outras_fontes/inse.parquet')

Próxima base - IVS

In [ ]:
ivs = pd.read_csv('../data/outras_fontes/ivs.xlsx')
ivs.head()

,NU_ANO_SAEB,CO_UF,SG_UF,NO_UF,CO_MUNICIPIO,NO_MUNICIPIO,TP_TIPO_REDE,TP_LOCALIZACAO,TP_CAPITAL,QTD_ALUNOS_INSE,MEDIA_INSE,PC_NIVEL_1,PC_NIVEL_2,PC_NIVEL_3,PC_NIVEL_4,PC_NIVEL_5,PC_NIVEL_6,PC_NIVEL_7,PC_NIVEL_8
0,2023,11,RO,Rondônia,1100122,Ji-Paraná,2,1,2,3258,"5,0083","0,16","6,71","18,43","25,63","24,64","14,36","9,18","0,89"
1,2023,11,RO,Rondônia,1101559,Teixeirópolis,0,2,2,19,"4,8242","5,26",NaN,"21,05","31,58","26,32","15,79",NaN,NaN
2,2023,11,RO,Rondônia,1100205,Porto Velho,3,1,1,3610,"4,8843","0,57","11,59",20,"26,07","20,78","12,87","7,11","1,02"
3,2023,11,RO,Rondônia,1101005,Governador Jorge Teixeira,5,2,2,46,"4,8115",0,"6,59","21,59","39,09","19,77","8,64","4,32",NaN
4,2023,11,RO,Rondônia,1100205,Porto Velho,2,1,1,9541,"5,0505","0,3","7,89","17,27","23,41","23,09","16,37","10,45","1,23"
